# ADNI No-MCI Dataset Cohort And Shape Audit

This notebook compares three ADNI hippocampus cohorts used in the longitudinal shape experiments:

- **Large ADNI no MCI:** the large strict-left no-MCI cohort before the later longitudinal QC filter.
- **QC-filtered large ADNI:** the filtered large cohort used by the current BrainODE/PCA/SIREN comparisons.
- **Old ADNI subset:** the smaller historical ADNI subset used by the previous BrainODE/PCA experiments.

The plots focus on cohort balance, longitudinal scan density, observed AD/CN atrophy, volume trends,
and representative ground-truth shapes. No model inference or mesh generation is performed here.


In [1]:
from pathlib import Path
import importlib
import sys

import pandas as pd
from IPython.display import Markdown, display

root = Path.cwd().resolve()
while not (root / ".git").exists():
    if root.parent == root:
        raise RuntimeError("Could not locate repo root from current working directory.")
    root = root.parent

script_dir = root / "examples" / "ADNI_1_L_No_MCI" / "brainode_comparison_task3_core_brainode_original" / "scripts"
if str(script_dir) not in sys.path:
    sys.path.insert(0, str(script_dir))

import dataset_cohort_visualization_support as dcv
dcv = importlib.reload(dcv)

frame = dcv.load_cohort_frame()
display(Markdown(
    f"Loaded `{len(frame)}` ground-truth scan rows from `{frame['subject_id'].nunique()}` unique subject IDs. "
    "Volumes are displayed in cubic centimeters. Meshes are only loaded for representative examples."
))


Loaded `5671` ground-truth scan rows from `920` unique subject IDs. Volumes are displayed in cubic centimeters. Meshes are only loaded for representative examples.

## Data Sources

The notebook reuses the existing whole-dataset volume audit table. For the old subset, the original audit
volume is converted back to physical scale using the median conversion factor observed in the large cohorts.
This keeps all plotted hippocampus volumes on the same `cm^3` scale.


In [2]:
display(dcv.data_sources_table())

overview = dcv.cohort_overview(frame)
display(Markdown(
    "### Cohort overview table\n"
    "This table counts unique scans and unique subjects per dataset and diagnosis. "
    "The atrophy columns summarize start-to-end subject-level volume change."
))
display(overview)


,dataset,display_name,description,primary_source
0,large_all,Large ADNI no MCI,All selected strict-left no-MCI ADNI large sca...,/home/jakaria/INR/Deep3DComp/examples/ADNI_1_L...
1,large_qc,QC-filtered large ADNI,QC-filtered large ADNI subset used by the curr...,/home/jakaria/INR/Deep3DComp/examples/ADNI_1_L...
2,old_small,Old ADNI subset,Older ADNI no-MCI subset used by the previous ...,/home/jakaria/INR/Deep3DComp/examples/ADNI_1_L...


### Cohort overview table
This table counts unique scans and unique subjects per dataset and diagnosis. The atrophy columns summarize start-to-end subject-level volume change.

,dataset,dataset_label,diagnosis,scans,subjects,mean_age_years,median_volume_cm3,avg_scans_per_subject,median_scans_per_subject,median_followup_years,mean_annual_percent_change,median_annual_percent_change
0,large_all,Large ADNI no MCI,AD,990,336,75.773939,2.386846,2.954955,3.0,1.0,-3.931140,-5.336880
1,large_all,Large ADNI no MCI,CN,1725,343,76.727839,3.382466,5.046647,5.0,3.0,2.195777,-0.748712
2,large_qc,QC-filtered large ADNI,AD,633,205,75.159507,2.470350,3.087805,3.0,1.0,-4.485779,-4.897026
3,large_qc,QC-filtered large ADNI,CN,1596,318,76.845084,3.382389,5.018868,5.0,3.0,-0.622497,-0.767934
4,old_small,Old ADNI subset,AD,302,101,76.347682,1.233044,2.990099,3.0,1.0,-5.914214,-5.546494
5,old_small,Old ADNI subset,CN,425,143,76.684706,1.680248,2.972028,3.0,1.0,-1.193764,-1.694889


## AD/CN Counts

This section answers how many AD and CN subjects and scans are available in each dataset. The first plot
uses all scans. The second plot splits subjects by train/val/test so model-evaluation imbalance is visible.


In [3]:
fig = dcv.plot_subject_and_scan_counts(frame)
fig.show()

display(Markdown(
    "The left panel is subject count and the right panel is scan count. "
    "A large gap between scans and subjects means subjects have repeated longitudinal visits."
))

fig = dcv.plot_split_counts(frame)
fig.show()

display(Markdown(
    "Train/val/test counts are shown by subject, with scan counts in hover text. "
    "This is useful before comparing longitudinal models because a split can have enough scans but few subjects."
))


The left panel is subject count and the right panel is scan count. A large gap between scans and subjects means subjects have repeated longitudinal visits.

Train/val/test counts are shown by subject, with scan counts in hover text. This is useful before comparing longitudinal models because a split can have enough scans but few subjects.

## Longitudinal Density

These plots show how many scans each subject contributes and how long the subject is followed. Better
longitudinal modeling needs both repeated visits and enough follow-up time.


In [4]:
fig = dcv.plot_scan_count_and_followup(frame)
fig.show()

scan_followup = dcv.subject_summary(frame).groupby(
    ["dataset_label", "diagnosis"],
    sort=False,
).agg(
    subjects=("subject_id", "nunique"),
    mean_scans=("scan_count", "mean"),
    median_scans=("scan_count", "median"),
    mean_followup_years=("followup_years", "mean"),
    median_followup_years=("followup_years", "median"),
).reset_index()
display(scan_followup)

display(Markdown(
    "The box centers and spread show whether the cohort is truly longitudinal. "
    "Subjects with one scan contribute to baseline distributions but not to start-to-end atrophy rates."
))


,dataset_label,diagnosis,subjects,mean_scans,median_scans,mean_followup_years,median_followup_years
0,Large ADNI no MCI,AD,333,2.954955,3.0,1.210210,1.0
1,Large ADNI no MCI,CN,343,5.046647,5.0,3.641399,3.0
2,QC-filtered large ADNI,AD,205,3.087805,3.0,1.424390,1.0
3,QC-filtered large ADNI,CN,318,5.018868,5.0,3.727987,3.0
4,Old ADNI subset,AD,101,2.990099,3.0,0.995050,1.0
5,Old ADNI subset,CN,143,2.972028,3.0,0.986014,1.0


The box centers and spread show whether the cohort is truly longitudinal. Subjects with one scan contribute to baseline distributions but not to start-to-end atrophy rates.

## Age And Baseline Volume

These figures check whether AD and CN have comparable baseline age and hippocampus volume. The volume
unit is `cm^3`, so typical hippocampus values should be a few cubic centimeters rather than thousands.


In [5]:
fig = dcv.plot_age_distribution(frame)
fig.show()

fig = dcv.plot_baseline_volume_and_atrophy(frame)
fig.show()

display(Markdown(
    "Baseline volume is measured at the first available scan for each subject. "
    "Atrophy rate is annualized from that subject's first to last observed scan."
))

display(dcv.atrophy_summary_table(frame))


Baseline volume is measured at the first available scan for each subject. Atrophy rate is annualized from that subject's first to last observed scan.

,dataset,dataset_label,diagnosis,subjects,subjects_with_followup,mean_followup_years,median_followup_years,mean_annual_percent_change,median_annual_percent_change,mean_annual_absolute_change_cm3
0,large_all,Large ADNI no MCI,AD,333,279,1.210210,1.0,-3.931140,-5.336880,-0.132166
1,large_all,Large ADNI no MCI,CN,343,329,3.641399,3.0,2.195777,-0.748712,-0.007197
2,large_qc,QC-filtered large ADNI,AD,205,205,1.424390,1.0,-4.485779,-4.897026,-0.116043
3,large_qc,QC-filtered large ADNI,CN,318,318,3.727987,3.0,-0.622497,-0.767934,-0.021047
4,old_small,Old ADNI subset,AD,101,101,0.995050,1.0,-5.914214,-5.546494,-0.075955
5,old_small,Old ADNI subset,CN,143,143,0.986014,1.0,-1.193764,-1.694889,-0.024517


## Observed Volume Trends

The relative trend plot normalizes each subject by their own baseline volume. It therefore compares
longitudinal change shape instead of absolute hippocampus size. The shaded band is the interquartile range
of subject curves at each follow-up time.


In [6]:
fig = dcv.plot_volume_trends(frame)
fig.show()

display(Markdown(
    "More negative relative change means stronger hippocampal volume loss after baseline. "
    "The plotted trend is observed data only; no forecast model is used."
))

fig = dcv.plot_volume_vs_age(frame)
fig.show()

display(Markdown(
    "The age plot uses each subject's baseline scan. "
    "It shows dataset-specific age/volume spread and whether AD/CN are naturally separated at baseline."
))


More negative relative change means stronger hippocampal volume loss after baseline. The plotted trend is observed data only; no forecast model is used.

The age plot uses each subject's baseline scan. It shows dataset-specific age/volume spread and whether AD/CN are naturally separated at baseline.

## QC Filter Impact

This section compares the large no-MCI cohort before and after the QC longitudinal filter. It shows how
many scans and subjects were retained or removed, separated by AD/CN diagnosis.


In [7]:
fig = dcv.plot_qc_filter_impact(frame)
fig.show()

display(dcv.qc_filter_impact_table(frame))

display(Markdown(
    "Removed rows are present in the large no-MCI cohort but not in the QC-filtered cohort. "
    "This is a dataset-processing diagnostic, not a model-quality metric."
))


,diagnosis,qc_status,scans,subjects,median_volume_cm3,median_age_years
0,AD,kept_in_qc,633,205,2.470350,75.800000
1,AD,removed_by_qc,357,164,2.220559,77.400000
2,CN,kept_in_qc,1596,318,3.382389,76.602396
3,CN,removed_by_qc,129,57,3.398613,75.234908


Removed rows are present in the large no-MCI cohort but not in the QC-filtered cohort. This is a dataset-processing diagnostic, not a model-quality metric.

## Representative Ground-Truth Shapes

The mesh grid shows one baseline shape per dataset for:

- CN near the dataset's median baseline volume.
- AD near the dataset's median baseline volume.
- CN with the largest baseline volume.
- CN with the smallest baseline volume.

Meshes are loaded from the existing ground-truth mesh paths and displayed as solid surfaces. If a mesh load
fails, that panel is skipped and the figure title reports the skipped count.


In [8]:
examples = dcv.shape_examples_table(frame)
display(examples)

fig = dcv.plot_shape_examples(frame)
fig.show()

display(Markdown(
    "The table gives the exact subject, scan, split, age, volume, and mesh path used in each panel. "
    "These meshes are visual examples only; the cohort statistics above are computed from all available audit rows."
))


,dataset,dataset_label,example,diagnosis,subject_id,scan_id,split,age_years,volume_cm3,mesh_path
0,large_all,Large ADNI no MCI,CN median baseline volume,CN,441,441_bl_left,train,72.70000,3.448220,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
1,large_all,Large ADNI no MCI,CN large baseline volume,CN,419,419_bl_left,train,70.20000,5.697878,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
2,large_all,Large ADNI no MCI,CN small baseline volume,CN,4951,4951_bl_left,train,71.90000,0.417046,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
3,large_all,Large ADNI no MCI,AD median baseline volume,AD,4192,4192_bl_left,train,82.20000,2.503048,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
4,large_qc,QC-filtered large ADNI,CN median baseline volume,CN,312,312_bl_left,train,82.80000,3.449572,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
5,large_qc,QC-filtered large ADNI,CN large baseline volume,CN,4520,4520_bl_left,train,67.80000,4.757304,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
6,large_qc,QC-filtered large ADNI,CN small baseline volume,CN,118,118_m12_left,train,81.39384,2.136305,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
7,large_qc,QC-filtered large ADNI,AD median baseline volume,AD,743,743_bl_left,train,85.60000,2.571241,/home/jakaria/ADNI/ADNI_1_GO_Large/left_hippoc...
8,old_small,Old ADNI subset,CN median baseline volume,CN,130_S_0232,ADNI_130_S_0232_MR_Hippocampal_Mask_Hi_2008030...,train,78.00000,1.692432,/home/jakaria/ADNI/ADNI_1/adni_processed/left_...
9,old_small,Old ADNI subset,CN large baseline volume,CN,037_S_0327,ADNI_037_S_0327_MR_Hippocampal_Mask_Hi_2008040...,train,70.00000,2.321311,/home/jakaria/ADNI/ADNI_1/adni_processed/left_...


The table gives the exact subject, scan, split, age, volume, and mesh path used in each panel. These meshes are visual examples only; the cohort statistics above are computed from all available audit rows.

## Dataset-Specific Checks

This final cell prints compact dataset summaries that are useful when deciding which cohort is suitable for
longitudinal experiments. Use it to quickly compare scan count, follow-up length, baseline volume scale,
and observed AD/CN annualized change.


In [9]:
subjects = dcv.subject_summary(frame)
for dataset in dcv.DATASET_ORDER:
    subset = subjects.loc[subjects["dataset"].eq(dataset)].copy()
    display(Markdown(f"### {dcv.DATASET_LABELS[dataset]}"))
    display(
        subset.groupby(["split", "diagnosis"], sort=False).agg(
            subjects=("subject_id", "nunique"),
            mean_scans=("scan_count", "mean"),
            median_followup_years=("followup_years", "median"),
            median_baseline_volume_cm3=("baseline_volume_cm3", "median"),
            median_annual_percent_change=("annual_percent_change", "median"),
        ).reset_index()
    )


### Large ADNI no MCI

,split,diagnosis,subjects,mean_scans,median_followup_years,median_baseline_volume_cm3,median_annual_percent_change
0,test,AD,38,2.868421,1.0,2.495762,-5.155198
1,test,CN,30,5.200000,3.0,3.279663,-0.961345
2,train,AD,256,2.984375,1.0,2.443717,-5.214536
3,train,CN,284,4.996479,3.0,3.448896,-0.628829
4,val,AD,39,2.846154,1.0,2.759717,-6.968880
5,val,CN,29,5.379310,3.0,3.600226,-1.063665


### QC-filtered large ADNI

,split,diagnosis,subjects,mean_scans,median_followup_years,median_baseline_volume_cm3,median_annual_percent_change
0,test,AD,21,3.095238,2.0,2.473282,-3.492717
1,test,CN,29,5.137931,3.0,3.284488,-1.038744
2,train,AD,160,3.100000,2.0,2.562119,-4.819864
3,train,CN,261,4.961686,3.0,3.449572,-0.626456
4,val,AD,24,3.000000,1.0,2.828187,-6.817158
5,val,CN,28,5.428571,3.0,3.606143,-1.107653


### Old ADNI subset

,split,diagnosis,subjects,mean_scans,median_followup_years,median_baseline_volume_cm3,median_annual_percent_change
0,test,AD,9,2.888889,1.0,1.374692,-4.213041
1,test,CN,15,3.000000,1.0,1.670207,-0.104058
2,train,AD,88,3.000000,1.0,1.278972,-5.446915
3,train,CN,119,2.966387,1.0,1.701916,-2.062806
4,val,AD,4,3.000000,1.0,1.175356,-8.468960
5,val,CN,9,3.000000,1.0,1.670557,-1.694889
